# New stuff in `pprop`: non-parametrized gates, shared parameters, pruning, and controlled rotations

This is a short tour of features that aren't covered by the other tutorial notebooks

## Imports

In [ ]:
import numpy as np
import pennylane as qml
from pennylane import numpy as pnp

from pprop import Propagator
from pprop.propagator.binding import Free


## 1. Non-parametrized gates

You can now assign value to parametrized gates upon defining the Ansatz.

**Note**: DO NOT assing int values, as they will be parsed as indexes of the param array

In [ ]:
def circuit_gates_demo(params):
    qml.RX(params[0], wires=0)
    qml.Hadamard(wires=1)
    qml.S(wires=2)
    qml.SX(wires=0)
    qml.T(wires=1)
    qml.RY(2., wires=2) # NOTE: pass 2 as a float (2.), otherwise the outputs
                        # will be different from Pennylanes
    qml.CNOT(wires=[0, 1])
    qml.CY(wires=[1, 2])
    qml.CZ(wires=[2, 0])
    qml.SWAP(wires=[0, 2])
    qml.RZ(np.pi, wires=1) # NOTE: np.pi is fine
    return [qml.expval(qml.PauliZ(0) @ qml.PauliZ(1)), qml.expval(qml.PauliX(2))]

In [ ]:
prop_gates = Propagator(circuit_gates_demo, k1=None, k2=None)
prop_gates.propagate()

rng = np.random.default_rng(0)
params = rng.random(prop_gates.num_params)

e_pprop = prop_gates(params)

dev = qml.device("default.qubit", wires=3)
qnode = qml.QNode(circuit_gates_demo, dev)
e_qml = np.array(qnode(params))

print(f"pprop : {e_pprop}")
print(f"qml   : {e_qml}")
assert np.allclose(e_pprop, e_qml, atol=1e-10), "Mismatch between pprop and PennyLane"
print("\nOutputs match.")

## 2. Multiple gates sharing the same parameter

A gate's `parameter` is just an integer *index* into the trainable-parameter vector, nothing requires that index to be unique. Two (or more) gates can reference the same index, tying them together: they always rotate by the same angle, and their gradient contributions with respect to that shared parameter simply add up.

`Propagator.num_params` reflects this correctly: it's inferred as `max(parameter_indices) + 1`, **not** the number of parametrized gates in the circuit. In the example below there are 4 rotation gates but only 2 distinct parameters.

In [ ]:
def circuit_shared(params):
    qml.RX(params[0], wires=0)
    qml.RY(params[0], wires=1)   # tied to the same parameter as the RX above
    qml.CNOT(wires=[0, 1])
    qml.RZ(params[1], wires=1)
    qml.RX(params[1], wires=0)   # tied to the same parameter as the RZ above
    qml.CNOT(wires=[1, 0])
    return qml.expval(qml.PauliZ(0) @ qml.PauliZ(1))

In [ ]:
prop_shared = Propagator(circuit_shared)
prop_shared.propagate()

n_param_gates = sum(1 for g in prop_shared.gates if g.parameter is not None)
print(f"Parametrized gates : {n_param_gates}")
print(f"Distinct parameters: {prop_shared.num_params}")

We now check both the energy and the gradient against PennyLane's own parameter-shift differentiation of the identical (tied) circuit. PennyLane accumulates the gradient of a repeated parameter the same way, so this is a meaningful cross-check, not just a self-consistency check within `pprop`.

In [ ]:
params2 = pnp.array(rng.random(prop_shared.num_params), requires_grad=True)

dev2 = qml.device("default.qubit", wires=2)
qnode2 = qml.QNode(circuit_shared, dev2, diff_method="parameter-shift")

e_qml2 = qnode2(params2)
grad_qml2 = qml.grad(qnode2)(params2)

e_pprop2, grad_pprop2 = prop_shared.eval_and_grad(np.array(params2))

print(f"pprop energy   : {e_pprop2[0]:.10f}")
print(f"qml energy     : {float(e_qml2):.10f}")
print(f"pprop gradient : {grad_pprop2[0]}")
print(f"qml gradient   : {np.array(grad_qml2)}")

assert np.isclose(e_pprop2[0], e_qml2, atol=1e-10)
assert np.allclose(grad_pprop2[0], np.array(grad_qml2), atol=1e-8)
print("\nOutputs and gradients match.")

### 2.1. More complex relationships between parameters

Plain index sharing ties gates to *exactly* the same value. `Propagator.bind()` goes further: it reparametrises the whole `num_params`-sized vector as an **affine** function of a smaller set of free parameters, so one gate can read `f0` and another `-2 * f0`, or a combination of several free parameters plus a constant offset.

`Free.vars(n)` gives you `n` symbols; combine them with `+`, `-`, `*`, `/` and plain numbers exactly as you'd write the relationship by hand, then pass one such expression per trainable index (same order as the ansatz) to `prop.bind(...)`.

Below, 4 gates depend on only 2 free parameters: `f0`, `-2*f0`, `f1`, and `f0 + 3*f1 - 0.5`.

In [ ]:
def circuit_bind_demo(params):
    qml.Hadamard(wires=0)
    qml.RY(params[0], wires=0)   # will represent: f0
    qml.RX(params[1], wires=1)   # will represent: -2 * f0
    qml.CNOT(wires=[0, 1])
    qml.RZ(params[2], wires=1)   # will represent: f1
    qml.RY(params[3], wires=0)   # will represent: f0 + 3*f1 - 0.5
    return qml.expval(qml.PauliZ(0) @ qml.PauliZ(1))


def free_circuit_bind_demo(free):
    # A plain PennyLane circuit written directly in terms of the free
    # parameters, used only as the ground truth to check against below.
    qml.Hadamard(wires=0)
    qml.RY(free[0], wires=0)
    qml.RX(-2 * free[0], wires=1)
    qml.CNOT(wires=[0, 1])
    qml.RZ(free[1], wires=1)
    qml.RY(free[0] + 3 * free[1] - 0.5, wires=0)
    return qml.expval(qml.PauliZ(0) @ qml.PauliZ(1))

In [ ]:
prop_bind = Propagator(circuit_bind_demo)
prop_bind.propagate()

f0, f1 = Free.vars(2)
bound = prop_bind.bind([f0, -2 * f0, f1, f0 + 3 * f1 - 0.5])
print(f"Parametrized gates : {prop_bind.num_params}")
print(f"Free parameters    : {bound.num_free}")

dev = qml.device("default.qubit", wires=2)
qnode_bind = qml.QNode(free_circuit_bind_demo, dev, diff_method="parameter-shift")

free_true = pnp.array(rng.uniform(-2, 2, 2), requires_grad=True)

e_qml = qnode_bind(free_true)
grad_qml = qml.grad(qnode_bind)(free_true)

e_pprop, grad_pprop = bound.eval_and_grad(np.array(free_true))

print(f"pprop energy   : {e_pprop[0]:.10f}")
print(f"qml energy     : {float(e_qml):.10f}")
print(f"pprop gradient : {grad_pprop[0]}")
print(f"qml gradient   : {np.array(grad_qml)}")

assert np.isclose(e_pprop[0], e_qml, atol=1e-8)
assert np.allclose(grad_pprop[0], np.array(grad_qml), atol=1e-8)
print("\nOutputs and gradients match.")

## 3. Pruning

`pprop`'s Rust backend offers two **exact** pruning strategies, enabled as flags on `.propagate()`. They never change the result, only the time it takes to get there:

- **`use_dead_qubit_pruner=True`**: drops any Pauli word carrying an $X$/$Y$ on a qubit no remaining gate can ever touch again.
- **`use_xy_weight_pruner=True`**: drops any Pauli word whose XY-weight exceeds the maximum reduction the remaining circuit could still achieve.

In [ ]:
import time

side = 6
J, h = 1.0, 1.0
num_qubits = side * side


def hamiltonian(side, J, h):
    coeffs, obs = [], []
    N = side * side
    for x in range(side):
        for y in range(side):
            i = x * side + y
            if y < side - 1:
                j = x * side + (y + 1)
                coeffs.append(-J / N)
                obs.append(qml.PauliZ(i) @ qml.PauliZ(j))
            if x < side - 1:
                j = (x + 1) * side + y
                coeffs.append(-J / N)
                obs.append(qml.PauliZ(i) @ qml.PauliZ(j))
    for i in range(N):
        coeffs.append(-h / N)
        obs.append(qml.PauliX(i))
    return qml.Hamiltonian(coeffs, obs)


def circuit_pruning_demo(params):
    index = 0
    for q in range(num_qubits):
        qml.RX(params[index], wires=q); index += 1
    for d in range(3):
        y_start = 0 if d % 2 == 0 else 1
        for x in range(side):
            for y in range(y_start, side - 1, 2):
                qml.CNOT(wires=[x * side + y, x * side + (y + 1)])
        for q in range(num_qubits):
            qml.RY(params[index], wires=q); index += 1
    return qml.expval(hamiltonian(side, J, h))

In [ ]:
prop_no_opt = Propagator(circuit_pruning_demo)
t0 = time.perf_counter()
prop_no_opt.propagate()
t_no_opt = time.perf_counter() - t0

prop_opt = Propagator(circuit_pruning_demo)
t0 = time.perf_counter()
prop_opt.propagate(use_dead_qubit_pruner=True, use_xy_weight_pruner=True)
t_opt = time.perf_counter() - t0

params3 = rng.random(prop_no_opt.num_params)
e_no_opt = prop_no_opt(params3)[0]
e_opt = prop_opt(params3)[0]

print(f"Time without pruning : {t_no_opt:.3f} s")
print(f"Time with pruning    : {t_opt:.3f} s")
print(f"Speedup              : {t_no_opt / t_opt:.2f}x")

assert np.isclose(e_no_opt, e_opt, atol=1e-10), "Pruning changed the result"
print("\nOutputs match.")

## 4. Controlled rotations and the θ/2 convention

  `pprop`'s controlled rotation gates (`CRX`, `CRY`, `CRZ`) do not use the
  same angle convention as PennyLane's. 
  
  A plain `RX(θ)`/`RY(θ)`/`RZ(θ)` matches
  PennyLane exactly, but a controlled rotation `C·RX(θ)` in `pprop`
  applies twice the rotation PennyLane would for the same `θ`: internally it
  generates the block as $\exp(-i\theta P)$ rather than PennyLane's
  $\exp(-i\theta/2 P)$.

  Concretely, to reproduce PennyLane's `CRX(θ)`/`CRY(θ)` in `pprop`, you must pass
  `θ/2` as the parameter value at evaluation time. This only affects the indices
  that feed controlled rotations, plain rotation gates sharing the same parameter
  vector are unaffected and should be left as is.

In [ ]:
def circuit_cr_demo(params):
    qml.Hadamard(wires=0)
    qml.RY(params[1], wires=1)
    qml.CRX(params[0], wires=[0, 1])
    qml.CRY(params[2], wires=[1, 2])
    return qml.expval(qml.PauliZ(1) @ qml.PauliX(0))


# Indices 0 and 2 feed CRX/CRY and need halving at eval time; index 1 (a
# plain RY) is used as-is.
half_angle_indices = [0, 2]

In [ ]:
dev = qml.device("default.qubit", wires=3)
qnode_cr = qml.QNode(circuit_cr_demo, dev, diff_method="parameter-shift")

prop_cr = Propagator(circuit_cr_demo)
prop_cr.propagate()

true_params = pnp.array(rng.uniform(-np.pi, np.pi, prop_cr.num_params), requires_grad=True)

e_qml = qnode_cr(true_params)
grad_qml = qml.grad(qnode_cr)(true_params)

eval_params = np.array(true_params, dtype=float)
eval_params[half_angle_indices] /= 2
e_pprop, grad_pprop = prop_cr.eval_and_grad(eval_params)

# Chain rule: eval_params[i] = true_params[i] / 2 at the half-angle indices,
# so d(output)/d(true_params[i]) = grad_pprop[i] / 2 there.
grad_pprop_true = np.array(grad_pprop[0])
grad_pprop_true[half_angle_indices] /= 2

print(f"pprop energy   : {e_pprop[0]:.10f}")
print(f"qml energy     : {float(e_qml):.10f}")
print(f"pprop gradient : {grad_pprop_true}")
print(f"qml gradient   : {np.array(grad_qml)}")

assert np.isclose(e_pprop[0], e_qml, atol=1e-8)
assert np.allclose(grad_pprop_true, np.array(grad_qml), atol=1e-8)
print("\nOutputs and gradients match.")